# MobileViT: Fine-tune a pretrained model for image classification

**Based on:** [Sayak Paul](https://twitter.com/RisingSayak), Keras MobileViT example<br>
**Description:** Fine-tune Hugging Face [`apple/mobilevit-small`](https://huggingface.co/apple/mobilevit-small) (ImageNet weights) on a downstream dataset instead of training a small model from scratch.

## Introduction

[MobileViT](https://arxiv.org/abs/2110.02178) ([Mehta et al.](https://arxiv.org/abs/2110.02178)) mixes convolutions and Transformers for efficient image classification. Here we **fine-tune** the official small checkpoint from the Hugging Face Hub rather than defining the architecture in Keras and training from random initialization.

**Requirements:** PyTorch (`torch`), `transformers`, `datasets`, and `accelerate` (for `Trainer`). Install if needed, for example: `pip install torch transformers datasets accelerate torchvision pillow evaluate`.

**Note:** In current `transformers` releases, image preprocessing is exposed as `AutoImageProcessor` / `MobileViTImageProcessor`; the older name `MobileViTFeatureExtractor` may not import. The notebook uses `AutoImageProcessor.from_pretrained("apple/mobilevit-small")`, which loads the same preprocessor config.

**Fine-tuning vs full training:** We load pretrained weights, replace the ImageNet classification head with `num_labels` for your task, and train with a **lower learning rate** than typical scratch training.

## Imports

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    MobileViTForImageClassification,
    Trainer,
    TrainingArguments,
)

feature_extractor = AutoImageProcessor.from_pretrained("apple/mobilevit-small")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Hyperparameters

In [ ]:
# Fine-tuning defaults (smaller LR than scratch training).
batch_size = 16
epochs = 5
learning_rate = 5e-5
weight_decay = 0.01

_sz = getattr(feature_extractor, "size", {}) or {}
if isinstance(_sz, dict):
    image_size = int(_sz.get("height", _sz.get("shortest_edge", 256)))
else:
    image_size = 256


## Dataset preparation

We use [`tf_flowers`](https://www.tensorflow.org/datasets/catalog/tf_flowers) via Hugging Face `datasets`. Images are resized and normalized by `MobileViTFeatureExtractor` (same preprocessing as pretraining).


In [ ]:
import tensorflow_datasets as tfds
from datasets import Dataset, ClassLabel
from PIL import Image as PILImage

tfds.disable_progress_bar()

(train_tf, val_tf), info = tfds.load(
    "tf_flowers",
    split=["train[:90%]", "train[90%:]"],
    as_supervised=True,
    with_info=True,
)

label_names = list(info.features["label"].names)


def tfds_split_to_hf(tfds_split):
    images, labels = [], []
    for image, label in tfds.as_numpy(tfds_split):
        images.append(PILImage.fromarray(image.astype("uint8")))
        labels.append(int(label))
    ds = Dataset.from_dict({"image": images, "label": labels})
    return ds.cast_column("label", ClassLabel(names=label_names))


train_ds = tfds_split_to_hf(train_tf)
val_ds = tfds_split_to_hf(val_tf)

labels = list(train_ds.features["label"].names)
num_classes = len(labels)
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}

print(f"Classes ({num_classes}): {labels}")


def preprocess_function(examples):
    images = [img.convert("RGB") for img in examples["image"]]
    # Fast image processors only support return_tensors="pt"; convert to NumPy for `datasets`.
    out = feature_extractor(images, return_tensors="pt")
    pixel_values = out["pixel_values"].detach().cpu().numpy()
    return {"pixel_values": pixel_values, "labels": examples["label"]}


train_ds = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
val_ds = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)

Dataset tf_flowers downloaded and prepared to /root/tensorflow_datasets/tf_flowers/3.0.1. Subsequent calls will reuse this data.


Casting the dataset:   0%|          | 0/3303 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

Classes (5): ['dandelion', 'daisy', 'tulips', 'sunflowers', 'roses']


Map:   0%|          | 0/3303 [00:00<?, ? examples/s]

Map:   0%|          | 0/367 [00:00<?, ? examples/s]

## Load pretrained `MobileViTForImageClassification`

Load ImageNet-pretrained weights and **replace the classifier head** for the flowers dataset (`num_labels=num_classes`, `ignore_mismatched_sizes=True`).



## Fine-tune with Hugging Face `Trainer`

We use a modest learning rate and weight decay typical for ViT-style fine-tuning.


In [ ]:
model = MobileViTForImageClassification.from_pretrained(
    "apple/mobilevit-small",
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/347 [00:00<?, ?it/s]

MobileViTForImageClassification LOAD REPORT from: apple/mobilevit-small
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([5])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 640]) vs model:torch.Size([5, 640])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters: 4,940,837


The next cell defines `compute_metrics` and builds `Trainer`. Install `evaluate` for the default path (`pip install evaluate`).


In [ ]:
try:
    import evaluate

    accuracy_metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return accuracy_metric.compute(predictions=predictions, references=labels)

except ImportError:
    from sklearn.metrics import accuracy_score

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {"accuracy": float(accuracy_score(labels, predictions))}


_base_args = dict(
    output_dir="./mobilevit-small-tf-flowers",
    remove_unused_columns=False,
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=weight_decay,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
)
try:
    training_args = TrainingArguments(**_base_args, eval_strategy="epoch")
except TypeError:
    training_args = TrainingArguments(**_base_args, evaluation_strategy="epoch")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


model.safetensors:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

## Train (fine-tune)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.852548,0.639414,0.904632
2,0.444649,0.297393,0.940054


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.852548,0.639414,0.904632
2,0.444649,0.297393,0.940054
3,0.363101,0.225196,0.950954
4,0.403861,0.185171,0.953678
5,0.206127,0.177843,0.956403


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1035, training_loss=0.5556377565227268, metrics={'train_runtime': 2522.7239, 'train_samples_per_second': 6.546, 'train_steps_per_second': 0.41, 'total_flos': 9.625682673598464e+16, 'train_loss': 0.5556377565227268, 'epoch': 5.0})

## Evaluate


In [ ]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.17784275114536285, 'eval_accuracy': 0.9564032697547684, 'eval_runtime': 46.8438, 'eval_samples_per_second': 7.835, 'eval_steps_per_second': 0.491, 'epoch': 5.0}


## Save the fine-tuned model

This PyTorch workflow saves weights and config in Hugging Face format.


In [ ]:
save_dir = "./mobilevit-small-tf-flowers-final"
trainer.save_model(save_dir)
feature_extractor.save_pretrained(save_dir)
print(f"Saved to {save_dir}")


Hub checkpoint used for fine-tuning: [apple/mobilevit-small](https://huggingface.co/apple/mobilevit-small).

In [ ]:
import shutil
import os

save_dir = "./mobilevit-small-tf-flowers-final"
drive_path = "/content/drive/MyDrive/mobilevit-small-tf-flowers-final"

# Ensure the parent directory in Google Drive exists
os.makedirs(os.path.dirname(drive_path), exist_ok=True)

# Copy the directory to Google Drive
shutil.copytree(save_dir, drive_path, dirs_exist_ok=True)

print(f"Successfully saved '{save_dir}' to '{drive_path}' in Google Drive.")

Successfully saved './mobilevit-small-tf-flowers-final' to '/content/drive/MyDrive/mobilevit-small-tf-flowers-final' in Google Drive.
